# Vote Type Comparison: Candidate Votes vs Proportional Votes

This notebook compares party support across Nepal Election 2082 using two datasets:

- `Exports/nepal_election_2082_results_ec.csv`: candidate-level constituency results
- `Exports/proportional_votes.csv`: nationwide proportional votes by party

The workflow below:

1. aggregates constituency candidate votes by party
2. aligns party names across the two files, including coalition-style shared symbols
3. compares absolute votes, vote shares, rank shifts, and relative performance
4. exports polished charts into `Images/`


In [1]:
from pathlib import Path
import importlib
import re
import subprocess
import sys
import textwrap
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import font_manager as fm

try:
    import kaleido  # noqa: F401
except ImportError:
    print('Installing missing package: kaleido')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'kaleido'])
    import kaleido  # noqa: F401

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == 'Codes' else CURRENT_DIR
EXPORT_DIR = PROJECT_DIR / 'Exports'
IMAGE_DIR = PROJECT_DIR / 'Images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

results_path = EXPORT_DIR / 'nepal_election_2082_results_ec.csv'
prop_path = EXPORT_DIR / 'proportional_votes.csv'

candidate_df = pd.read_csv(results_path)
proportional_df = pd.read_csv(prop_path)

font_candidates = [
    'Noto Sans Devanagari',
    'Kohinoor Devanagari',
    'Devanagari Sangam MN',
    'Devanagari MT',
    'Arial Unicode MS',
]
available_fonts = {font.name for font in fm.fontManager.ttflist}
selected_font = next((font for font in font_candidates if font in available_fonts), 'DejaVu Sans')

COLOR_FPTP = '#d95f02'
COLOR_PR = '#1b9e77'
COLOR_BG = '#f7f3ea'
COLOR_PANEL = '#fffdf8'
COLOR_TEXT = '#2f2a24'
COLOR_GRID = '#d9d2c7'
COLOR_POS = '#c44e52'
COLOR_NEG = '#4c72b0'
COLOR_MATCHED = '#6a994e'
COLOR_UNMATCHED = '#bc4749'

pio.templates.default = 'plotly_white'


SHORT_NAME_MAP = {
    'नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी लेनिनवादी)': 'नेकपा (एमाले)',
    'नेपाल कम्युनिष्ट पार्टी (माओवादी)': 'नेकपा (माओवादी)',
    'जनता समाजवादी पार्टी, नेपाल': 'जसपा नेपाल',
    'नेपाल मजदुर किसान पार्टी': 'नेमकिपा',
    'एकल चिन्ह मोबाइल (आम जनता पार्टी/जनादेश पार्टी नेपाल)': 'मोबाइल चिन्ह गठबन्धन',
    'एकल चिन्ह चकिया (जाँतो) (राष्ट्रिय मुक्ति पार्टी नेपाल/जनता समाजवादी पार्टी/नागरिक उन्मुक्ति पार्टी, नेपाल)': 'चकिया चिन्ह गठबन्धन',
    'एकल चिन्ह बस (नेपाल संघीय समाजवादी पार्टी/बहुजन एकता पार्टी नेपाल/नेपाल जनजागृति पार्टी)': 'बस चिन्ह गठबन्धन',
    'एकल चिन्ह जगर भएको सिंह(सचेत नेपाली पार्टी/नागरिक सर्वोच्चता पार्टी नेपाल)': 'जगर सिंह चिन्ह गठबन्धन',
}


def shorten_party_name(label, max_chars=30):
    if pd.isna(label):
        return label
    label = str(label).strip()
    label = SHORT_NAME_MAP.get(label, label)
    if len(label) <= max_chars:
        return label
    compact = label.replace(' पार्टी', '').replace(' नेपाल', '').replace(' राष्ट्रिय', ' रा.')
    if len(compact) <= max_chars:
        return compact
    return compact[: max_chars - 1].rstrip() + '…'


def wrap_label(label, width=18):
    if pd.isna(label):
        return label
    return '<br>'.join(textwrap.wrap(str(label), width=width, break_long_words=False, break_on_hyphens=False))


def style_figure(fig, height=850, width=1400, margin=None):
    fig.update_layout(
        height=height,
        width=width,
        paper_bgcolor=COLOR_BG,
        plot_bgcolor=COLOR_PANEL,
        font=dict(family=selected_font, size=16, color=COLOR_TEXT),
        margin=margin or dict(l=120, r=80, t=90, b=70),
        legend_title_text='',
    )
    fig.update_xaxes(showgrid=True, gridcolor=COLOR_GRID, zeroline=False, automargin=True, tickfont=dict(size=13))
    fig.update_yaxes(showgrid=True, gridcolor=COLOR_GRID, zeroline=False, automargin=True, tickfont=dict(size=13))
    return fig


def save_figure(fig, filename, scale=2):
    path = IMAGE_DIR / filename
    try:
        fig.write_image(path, scale=scale)
    except ValueError as exc:
        if 'kaleido' not in str(exc).lower():
            raise
        print('Refreshing Kaleido support for this kernel...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '--upgrade', 'kaleido'])
        import kaleido  # noqa: F401
        import plotly.io as _pio
        import plotly.io._kaleido as _pio_kaleido
        importlib.reload(_pio_kaleido)
        importlib.reload(_pio)
        fig.write_image(path, scale=scale)
    print(f'Exported {path.relative_to(PROJECT_DIR)}')
    return path

print(f'Loaded {candidate_df.shape[0]:,} candidate rows and {proportional_df.shape[0]:,} proportional rows.')
print(f'Using chart font: {selected_font}')


Loaded 3,406 candidate rows and 57 proportional rows.
Using chart font: Kohinoor Devanagari


## Party-name alignment logic

The two source files are not perfectly consistent on party naming. Some entries differ only by spelling, while some proportional votes are reported under a shared election symbol that corresponds to multiple constituency parties.

To keep the comparison valid, the notebook applies three steps:

- remove the `("एकल चुनाव चिन्ह")` suffix from constituency party names
- normalize a small set of known spelling differences
- map coalition members in constituency data to the combined proportional-vote label when the PR file uses a shared symbol


In [2]:
COALITION_MEMBER_TO_LABEL = {
    'आम जनता पार्टी': 'एकल चिन्ह मोबाइल (आम जनता पार्टी/जनादेश पार्टी नेपाल)',
    'जनादेश पार्टी नेपाल': 'एकल चिन्ह मोबाइल (आम जनता पार्टी/जनादेश पार्टी नेपाल)',
    'राष्ट्रिय मुक्ति पार्टी नेपाल': 'एकल चिन्ह चकिया (जाँतो) (राष्ट्रिय मुक्ति पार्टी नेपाल/जनता समाजवादी पार्टी/नागरिक उन्मुक्ति पार्टी, नेपाल)',
    'जनता समाजवादी पार्टी': 'एकल चिन्ह चकिया (जाँतो) (राष्ट्रिय मुक्ति पार्टी नेपाल/जनता समाजवादी पार्टी/नागरिक उन्मुक्ति पार्टी, नेपाल)',
    'नागरिक उन्मुक्ति पार्टी, नेपाल': 'एकल चिन्ह चकिया (जाँतो) (राष्ट्रिय मुक्ति पार्टी नेपाल/जनता समाजवादी पार्टी/नागरिक उन्मुक्ति पार्टी, नेपाल)',
    'नेपाल संघीय समाजवादी पार्टी': 'एकल चिन्ह बस (नेपाल संघीय समाजवादी पार्टी/बहुजन एकता पार्टी नेपाल/नेपाल जनजागृति पार्टी)',
    'बहुजन एकता पार्टी नेपाल': 'एकल चिन्ह बस (नेपाल संघीय समाजवादी पार्टी/बहुजन एकता पार्टी नेपाल/नेपाल जनजागृति पार्टी)',
    'नेपाल जनजागृति पार्टी': 'एकल चिन्ह बस (नेपाल संघीय समाजवादी पार्टी/बहुजन एकता पार्टी नेपाल/नेपाल जनजागृति पार्टी)',
    'सचेत नेपाली पार्टी': 'एकल चिन्ह जगर भएको सिंह(सचेत नेपाली पार्टी/नागरिक सर्वोच्चता पार्टी नेपाल)',
    'नागरिक सर्वोच्चता पार्टी नेपाल': 'एकल चिन्ह जगर भएको सिंह(सचेत नेपाली पार्टी/नागरिक सर्वोच्चता पार्टी नेपाल)',
}

ALIASES = {
    'त्रिमूल नेपाल': 'त्रिमुल नेपाल',
    'मंगोल नेशनल अर्गनाइजेसन': 'मंगोल नेशनल अर्गनाइजेशन',
    'उन्‍नत लोकतन्त्र पार्टी': 'उन्नत लोकतन्त्र पार्टी',
    'नेपाल सद्‍भावना पार्टी': 'नेपाल सद्भावना पार्टी',
}


def clean_party_name(name):
    if pd.isna(name):
        return np.nan
    cleaned = unicodedata.normalize('NFKC', str(name))
    cleaned = cleaned.replace('‍', '').replace('‌', '').replace(' ', ' ')
    cleaned = re.sub(r'\(एकल चुनाव चिन्ह\)', '', cleaned).strip()
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = ALIASES.get(cleaned, cleaned)
    cleaned = COALITION_MEMBER_TO_LABEL.get(cleaned, cleaned)
    return cleaned

candidate_df['party_clean'] = candidate_df['party_name_np'].map(clean_party_name)
proportional_df['party_clean'] = proportional_df['party_name'].map(clean_party_name)

fptp_party_votes = (
    candidate_df
    .groupby('party_clean', as_index=False)['votes']
    .sum()
    .rename(columns={'party_clean': 'party_name', 'votes': 'fptp_votes'})
)
pr_party_votes = (
    proportional_df
    .groupby('party_clean', as_index=False)['total_votes']
    .sum()
    .rename(columns={'party_clean': 'party_name', 'total_votes': 'pr_votes'})
)

comparison = fptp_party_votes.merge(pr_party_votes, on='party_name', how='outer', indicator=True)
comparison['fptp_votes'] = comparison['fptp_votes'].fillna(0).astype(int)
comparison['pr_votes'] = comparison['pr_votes'].fillna(0).astype(int)
comparison['vote_gap'] = comparison['fptp_votes'] - comparison['pr_votes']
comparison['gap_pct_vs_pr'] = np.where(comparison['pr_votes'] > 0, comparison['vote_gap'] / comparison['pr_votes'], np.nan)
comparison['fptp_to_pr_ratio'] = np.where(comparison['pr_votes'] > 0, comparison['fptp_votes'] / comparison['pr_votes'], np.nan)
comparison['fptp_share'] = comparison['fptp_votes'] / comparison['fptp_votes'].sum()
comparison['pr_share'] = comparison['pr_votes'] / comparison['pr_votes'].sum()
comparison['share_gap_pct_points'] = (comparison['fptp_share'] - comparison['pr_share']) * 100
comparison['matched'] = comparison['_merge'].eq('both')
comparison['party_short_name'] = comparison['party_name'].map(shorten_party_name)
comparison['party_label'] = comparison['party_short_name'].map(wrap_label)
comparison['party_full_label'] = comparison['party_name'].map(lambda x: wrap_label(x, width=24))

matched = comparison[comparison['matched']].copy()
unmatched_fptp = comparison[comparison['_merge'] == 'left_only'].copy()
unmatched_pr = comparison[comparison['_merge'] == 'right_only'].copy()

matched['combined_votes'] = matched['fptp_votes'] + matched['pr_votes']
matched['fptp_rank'] = matched['fptp_votes'].rank(method='dense', ascending=False).astype(int)
matched['pr_rank'] = matched['pr_votes'].rank(method='dense', ascending=False).astype(int)
matched['rank_shift'] = matched['pr_rank'] - matched['fptp_rank']
matched['party_short_name'] = matched['party_name'].map(shorten_party_name)
matched['party_label'] = matched['party_short_name'].map(wrap_label)
matched['party_full_label'] = matched['party_name'].map(lambda x: wrap_label(x, width=24))
matched = matched.sort_values('pr_votes', ascending=False).reset_index(drop=True)

comparison.head()


,party_name,fptp_votes,pr_votes,_merge,vote_gap,gap_pct_vs_pr,fptp_to_pr_ratio,fptp_share,pr_share,share_gap_pct_points,matched,party_short_name,party_label,party_full_label
0,इतिहासिक जनता पार्टी,0,0,left_only,0,NaN,NaN,0.000000,0.000000,0.000000,False,इतिहासिक जनता पार्टी,इतिहासिक जनता<br>पार्टी,इतिहासिक जनता पार्टी
1,उज्यालो नेपाल पार्टी,115569,0,left_only,115569,NaN,NaN,0.010989,0.000000,1.098949,False,उज्यालो नेपाल पार्टी,उज्यालो नेपाल<br>पार्टी,उज्यालो नेपाल पार्टी
2,उन्नत लोकतन्त्र पार्टी,0,730,right_only,-730,-1.000000,0.000000,0.000000,0.000067,-0.006737,False,उन्नत लोकतन्त्र पार्टी,उन्नत लोकतन्त्र<br>पार्टी,उन्नत लोकतन्त्र पार्टी
3,एकल चिन्ह चकिया (जाँतो) (राष्ट्रिय मुक्ति पार्...,49778,62069,both,-12291,-0.198022,0.801978,0.004733,0.005729,-0.099515,True,चकिया चिन्ह गठबन्धन,चकिया चिन्ह<br>गठबन्धन,एकल चिन्ह चकिया (जाँतो)<br>(राष्ट्रिय मुक्ति प...
4,एकल चिन्ह जगर भएको सिंह(सचेत नेपाली पार्टी/नाग...,120,1132,both,-1012,-0.893993,0.106007,0.000011,0.000104,-0.009307,True,जगर सिंह चिन्ह गठबन्धन,जगर सिंह चिन्ह<br>गठबन्धन,एकल चिन्ह जगर भएको<br>सिंह(सचेत नेपाली<br>पार्...


## Snapshot metrics

These metrics show how much of the original vote totals are covered after party-name alignment, plus a quick correlation check between the two vote systems.


In [3]:
summary = pd.DataFrame({
    'Metric': [
        'Candidate result rows',
        'Distinct constituency parties',
        'Distinct proportional parties',
        'Matched parties after cleaning',
        'PR vote coverage by matched parties',
        'Constituency vote coverage by matched parties',
        'Correlation between party vote totals',
        'Total unmatched constituency votes',
        'Total unmatched proportional votes',
    ],
    'Value': [
        f"{candidate_df.shape[0]:,}",
        f"{fptp_party_votes.shape[0]:,}",
        f"{pr_party_votes.shape[0]:,}",
        f"{matched.shape[0]:,}",
        f"{matched['pr_votes'].sum() / pr_party_votes['pr_votes'].sum():.2%}",
        f"{matched['fptp_votes'].sum() / fptp_party_votes['fptp_votes'].sum():.2%}",
        f"{matched['fptp_votes'].corr(matched['pr_votes']):.3f}",
        f"{unmatched_fptp['fptp_votes'].sum():,}",
        f"{unmatched_pr['pr_votes'].sum():,}",
    ],
})
display(summary)


,Metric,Value
0,Candidate result rows,"3,406"
1,Distinct constituency parties,64
2,Distinct proportional parties,57
3,Matched parties after cleaning,50
4,PR vote coverage by matched parties,99.83%
5,Constituency vote coverage by matched parties,97.90%
6,Correlation between party vote totals,0.994
7,Total unmatched constituency votes,"220,696"
8,Total unmatched proportional votes,"18,647"


## 1. Largest parties in both systems

This grouped bar chart shows the largest matched parties by total scale across the two voting systems. The x-axis range is padded so numeric labels remain visible in the exported PNG.


In [4]:
top_parties = matched.nlargest(12, 'combined_votes').copy()
top_plot = top_parties.melt(
    id_vars=['party_name', 'party_label'],
    value_vars=['fptp_votes', 'pr_votes'],
    var_name='vote_type',
    value_name='votes',
)
top_plot['vote_type'] = top_plot['vote_type'].map({
    'fptp_votes': 'Candidate votes (constituency)',
    'pr_votes': 'Proportional votes',
})
max_vote = top_plot['votes'].max()

fig = px.bar(
    top_plot,
    x='votes',
    y='party_label',
    color='vote_type',
    orientation='h',
    barmode='group',
    text='votes',
    color_discrete_map={
        'Candidate votes (constituency)': COLOR_FPTP,
        'Proportional votes': COLOR_PR,
    },
    title='Largest parties: constituency candidate votes vs proportional votes',
    hover_name='party_name',
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside', cliponaxis=False)
style_figure(fig, height=980, width=1600, margin=dict(l=340, r=190, t=90, b=60))
fig.update_xaxes(title='Votes', tickformat=',d', range=[0, max_vote * 1.22])
fig.update_yaxes(title='', categoryorder='total ascending')
save_figure(fig, 'vote_type_top_parties.png')
fig.show()


Exported Images/vote_type_top_parties.png


## 2. Relative strength by party

A ratio above `1.0` means a party collected more constituency candidate votes than proportional votes. A ratio below `1.0` means its nationwide proportional total was stronger than its constituency aggregation.


In [5]:
ratio_candidates = matched[matched['pr_votes'] >= 10000].sort_values('fptp_to_pr_ratio').copy()
ratio_view = pd.concat([ratio_candidates.head(8), ratio_candidates.tail(8)]).drop_duplicates().sort_values('fptp_to_pr_ratio')
ratio_max = ratio_view['fptp_to_pr_ratio'].max()
ratio_view['ratio_text'] = ratio_view['fptp_to_pr_ratio'].map(lambda x: f'{x:.2f}x')
ratio_view['direction'] = np.where(ratio_view['fptp_to_pr_ratio'] >= 1, 'FPTP stronger', 'PR stronger')

fig = px.bar(
    ratio_view,
    x='fptp_to_pr_ratio',
    y='party_label',
    color='direction',
    orientation='h',
    text='ratio_text',
    color_discrete_map={'FPTP stronger': COLOR_POS, 'PR stronger': COLOR_NEG},
    title='Relative strength by party: strongest FPTP vs strongest PR performers',
    hover_name='party_name',
)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.add_vline(x=1.0, line_dash='dash', line_color=COLOR_TEXT, line_width=2)
style_figure(fig, height=1180, width=1600, margin=dict(l=360, r=190, t=90, b=60))
fig.update_xaxes(title='FPTP / PR vote ratio', tickformat='.2f', range=[0, ratio_max * 1.18])
fig.update_yaxes(title='')
save_figure(fig, 'vote_type_ratio_by_party.png')
fig.show()


Exported Images/vote_type_ratio_by_party.png


## 3. Relationship between absolute vote totals

The scatter plot shows whether parties with high proportional support also have high constituency vote totals. A fitted line is added directly from the data, without relying on any extra regression package.


In [6]:
scatter_df = matched.copy()
scatter_df['label_text'] = np.where(scatter_df[['fptp_votes', 'pr_votes']].max(axis=1) >= 150000, scatter_df['party_name'], '')

slope, intercept = np.polyfit(scatter_df['pr_votes'], scatter_df['fptp_votes'], 1)
line_x = np.linspace(scatter_df['pr_votes'].min(), scatter_df['pr_votes'].max(), 200)
line_y = slope * line_x + intercept

fig = px.scatter(
    scatter_df,
    x='pr_votes',
    y='fptp_votes',
    text='label_text',
    color_discrete_sequence=['#2a6f97'],
    title='Party-level relationship between proportional votes and constituency candidate votes',
)
fig.update_traces(marker=dict(size=15, opacity=0.82), textposition='top center')
fig.add_trace(go.Scatter(x=line_x, y=line_y, mode='lines', name='Trend line', line=dict(color='#f4a261', width=3)))
style_figure(fig, height=940, width=1300, margin=dict(l=90, r=50, t=90, b=80))
fig.update_xaxes(title='Proportional votes', tickformat=',d')
fig.update_yaxes(title='Candidate votes (constituency total)', tickformat=',d')
save_figure(fig, 'vote_type_scatter_relationship.png')
fig.show()


Exported Images/vote_type_scatter_relationship.png


## 4. Vote-share comparison for the biggest parties

Absolute votes can be dominated by the largest parties, so this chart compares vote shares. The dumbbell format makes it easier to see whether a party gains or loses relative weight when moving from proportional votes to aggregated constituency votes.


In [7]:
share_view = matched.nlargest(10, 'combined_votes').sort_values('pr_share').copy()
share_max = max(share_view['pr_share'].max(), share_view['fptp_share'].max()) * 100

fig = go.Figure()
for _, row in share_view.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['pr_share'] * 100, row['fptp_share'] * 100],
        y=[row['party_label'], row['party_label']],
        mode='lines',
        line=dict(color='#b8b1a6', width=4),
        showlegend=False,
        hoverinfo='skip',
    ))
fig.add_trace(go.Scatter(
    x=share_view['pr_share'] * 100,
    y=share_view['party_label'],
    mode='markers',
    name='Proportional share',
    marker=dict(color=COLOR_PR, size=14),
    customdata=share_view['party_name'],
    hovertemplate='%{customdata}<br>PR share: %{x:.2f}%<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=share_view['fptp_share'] * 100,
    y=share_view['party_label'],
    mode='markers',
    name='Constituency share',
    marker=dict(color=COLOR_FPTP, size=14),
    customdata=share_view['party_name'],
    hovertemplate='%{customdata}<br>FPTP share: %{x:.2f}%<extra></extra>',
))
style_figure(fig, height=920, width=1550, margin=dict(l=340, r=130, t=90, b=60))
fig.update_layout(title='Vote-share comparison for the biggest matched parties')
fig.update_xaxes(title='Vote share (%)', range=[0, share_max * 1.12], ticksuffix='%')
fig.update_yaxes(title='')
save_figure(fig, 'vote_type_share_dumbbell.png')
fig.show()


Exported Images/vote_type_share_dumbbell.png


## 5. Rank shifts between the two systems

This chart highlights which parties move up or down when you rank them by proportional votes instead of constituency vote totals.

- movement to the left means the party ranks better in proportional votes
- movement to the right means it ranks better in constituency results


In [8]:
rank_view = matched.nsmallest(12, 'pr_rank').sort_values('pr_rank').copy()
max_rank = int(max(rank_view['pr_rank'].max(), rank_view['fptp_rank'].max()))

fig = go.Figure()
for _, row in rank_view.iterrows():
    fig.add_trace(go.Scatter(
        x=[row['pr_rank'], row['fptp_rank']],
        y=[row['party_label'], row['party_label']],
        mode='lines',
        line=dict(color='#c9c2b8', width=4),
        showlegend=False,
        hoverinfo='skip',
    ))
fig.add_trace(go.Scatter(
    x=rank_view['pr_rank'],
    y=rank_view['party_label'],
    mode='markers',
    name='PR rank',
    marker=dict(color=COLOR_PR, size=14),
    customdata=rank_view['party_name'],
    hovertemplate='%{customdata}<br>PR rank: %{x}<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=rank_view['fptp_rank'],
    y=rank_view['party_label'],
    mode='markers',
    name='FPTP rank',
    marker=dict(color=COLOR_FPTP, size=14),
    customdata=rank_view['party_name'],
    hovertemplate='%{customdata}<br>FPTP rank: %{x}<extra></extra>',
))
style_figure(fig, height=980, width=1550, margin=dict(l=350, r=130, t=90, b=60))
fig.update_layout(title='Rank shifts between proportional votes and constituency vote totals')
fig.update_xaxes(title='Rank position', range=[0.5, max_rank + 0.8], dtick=1)
fig.update_yaxes(title='')
save_figure(fig, 'vote_type_rank_shift.png')
fig.show()


Exported Images/vote_type_rank_shift.png


## 6. Coverage of the matching step

This figure shows how much of each source file ends up in the direct party-to-party comparison after cleaning.


In [9]:
coverage = pd.DataFrame({
    'dataset': ['Constituency votes', 'Constituency votes', 'Proportional votes', 'Proportional votes'],
    'status': ['Matched parties', 'Unmatched parties', 'Matched parties', 'Unmatched parties'],
    'votes': [
        matched['fptp_votes'].sum(),
        unmatched_fptp['fptp_votes'].sum(),
        matched['pr_votes'].sum(),
        unmatched_pr['pr_votes'].sum(),
    ],
})
coverage['votes_text'] = coverage['votes'].map(lambda x: f'{x:,}')
coverage_max = coverage['votes'].max()

fig = px.bar(
    coverage,
    x='dataset',
    y='votes',
    color='status',
    barmode='group',
    text='votes_text',
    color_discrete_map={'Matched parties': COLOR_MATCHED, 'Unmatched parties': COLOR_UNMATCHED},
    title='How much of each dataset is covered by the party-name matching step',
)
fig.update_traces(textposition='outside', cliponaxis=False)
style_figure(fig, height=780, width=1200, margin=dict(l=90, r=70, t=90, b=70))
fig.update_xaxes(title='')
fig.update_yaxes(title='Votes', tickformat=',d', range=[0, coverage_max * 1.18])
save_figure(fig, 'vote_type_matching_coverage.png')
fig.show()


Exported Images/vote_type_matching_coverage.png


## 7. Biggest positive and negative share shifts

This focuses on share change rather than raw vote totals. Positive values mean a party's share is higher in constituency vote totals than in proportional votes.


In [10]:
gap_view = matched[matched['pr_votes'] >= 5000].copy()
gap_view = gap_view.sort_values('share_gap_pct_points')
focus_gap = pd.concat([gap_view.head(7), gap_view.tail(7)]).drop_duplicates().sort_values('share_gap_pct_points')
focus_gap['gap_text'] = focus_gap['share_gap_pct_points'].map(lambda x: f'{x:+.2f} pp')
focus_gap['direction'] = np.where(focus_gap['share_gap_pct_points'] >= 0, 'Higher in constituency', 'Higher in proportional')
gap_extent = max(abs(focus_gap['share_gap_pct_points'].min()), abs(focus_gap['share_gap_pct_points'].max()))

fig = px.bar(
    focus_gap,
    x='share_gap_pct_points',
    y='party_label',
    color='direction',
    orientation='h',
    text='gap_text',
    color_discrete_map={'Higher in constituency': COLOR_POS, 'Higher in proportional': COLOR_NEG},
    title='Largest shifts in vote share between the two systems',
    hover_name='party_name',
)
fig.update_traces(textposition='outside', cliponaxis=False)
fig.add_vline(x=0, line_dash='dash', line_color=COLOR_TEXT, line_width=2)
style_figure(fig, height=1080, width=1600, margin=dict(l=360, r=190, t=90, b=60))
fig.update_xaxes(title='Share difference in percentage points (FPTP share - PR share)', range=[-gap_extent * 1.35, gap_extent * 1.35])
fig.update_yaxes(title='')
save_figure(fig, 'vote_type_share_shift.png')
fig.show()


Exported Images/vote_type_share_shift.png


## Detailed tables

The tables below make it easier to inspect the biggest differences, including the parties that still remain unmatched after cleaning.


In [11]:
largest_gaps = matched.loc[:, ['party_name', 'fptp_votes', 'pr_votes', 'vote_gap', 'gap_pct_vs_pr', 'fptp_to_pr_ratio', 'share_gap_pct_points']].copy()
largest_gaps = largest_gaps.sort_values('vote_gap', key=lambda s: s.abs(), ascending=False).head(20)
display(largest_gaps.style.format({
    'fptp_votes': '{:,.0f}',
    'pr_votes': '{:,.0f}',
    'vote_gap': '{:+,.0f}',
    'gap_pct_vs_pr': '{:+.1%}',
    'fptp_to_pr_ratio': '{:.2f}x',
    'share_gap_pct_points': '{:+.2f} pp',
}))


,party_name,fptp_votes,pr_votes,vote_gap,gap_pct_vs_pr,fptp_to_pr_ratio,share_gap_pct_points
0,राष्ट्रिय स्वतन्त्र पार्टी,"4,650,500","5,183,493","-532,993",-10.3%,0.90x,-3.62 pp
1,नेपाली काँग्रेस,"2,008,619","1,759,172","+249,447",+14.2%,1.14x,+2.86 pp
2,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी लेनिनवादी),"1,623,014","1,455,885","+167,129",+11.5%,1.11x,+2.00 pp
3,नेपाली कम्युनिष्ट पार्टी,"975,813","811,577","+164,236",+20.2%,1.20x,+1.79 pp
7,राष्ट्रिय परिवर्तन पार्टी,"10,006","172,489","-162,483",-94.2%,0.06x,-1.50 pp
5,राष्ट्रिय प्रजातन्त्र पार्टी,"203,123","330,684","-127,561",-38.6%,0.61x,-1.12 pp
4,श्रम संस्कृति पार्टी,"303,872","385,902","-82,030",-21.3%,0.79x,-0.67 pp
11,राष्ट्र निर्माण दल नेपाल,"5,030","39,577","-34,547",-87.3%,0.13x,-0.32 pp
14,नेपाल जनता संरक्षण पार्टी,588,"28,424","-27,836",-97.9%,0.02x,-0.26 pp
13,एकल चिन्ह बस (नेपाल संघीय समाजवादी पार्टी/बहुजन एकता पार्टी नेपाल/नेपाल जनजागृति पार्टी),"10,045","29,436","-19,391",-65.9%,0.34x,-0.18 pp


In [12]:
rank_table = matched.loc[:, ['party_name', 'pr_rank', 'fptp_rank', 'rank_shift', 'pr_share', 'fptp_share']].copy()
rank_table = rank_table.sort_values(['pr_rank', 'fptp_rank']).head(20)
display(rank_table.style.format({
    'pr_share': '{:.2%}',
    'fptp_share': '{:.2%}',
    'rank_shift': '{:+.0f}',
}))


,party_name,pr_rank,fptp_rank,rank_shift,pr_share,fptp_share
0,राष्ट्रिय स्वतन्त्र पार्टी,1,1,+0,47.84%,44.22%
1,नेपाली काँग्रेस,2,2,+0,16.24%,19.10%
2,नेपाल कम्युनिष्ट पार्टी (एकीकृत मार्क्सवादी लेनिनवादी),3,3,+0,13.44%,15.43%
3,नेपाली कम्युनिष्ट पार्टी,4,4,+0,7.49%,9.28%
4,श्रम संस्कृति पार्टी,5,5,+0,3.56%,2.89%
5,राष्ट्रिय प्रजातन्त्र पार्टी,6,6,+0,3.05%,1.93%
6,"जनता समाजवादी पार्टी, नेपाल",7,7,+0,1.68%,1.80%
7,राष्ट्रिय परिवर्तन पार्टी,8,18,-10,1.59%,0.10%
8,जनमत पार्टी,9,8,+1,0.73%,0.58%
9,"एकल चिन्ह चकिया (जाँतो) (राष्ट्रिय मुक्ति पार्टी नेपाल/जनता समाजवादी पार्टी/नागरिक उन्मुक्ति पार्टी, नेपाल)",10,9,+1,0.57%,0.47%


In [13]:
unmatched_table = pd.concat([
    unmatched_fptp.sort_values('fptp_votes', ascending=False).assign(source='Only in constituency results').head(10),
    unmatched_pr.sort_values('pr_votes', ascending=False).assign(source='Only in proportional results').head(10),
], ignore_index=True)
display(unmatched_table[['source', 'party_name', 'fptp_votes', 'pr_votes']].fillna(0).style.format({
    'fptp_votes': '{:,.0f}',
    'pr_votes': '{:,.0f}',
}))


,source,party_name,fptp_votes,pr_votes
0,Only in constituency results,उज्यालो नेपाल पार्टी,"115,569",0
1,Only in constituency results,स्वतन्त्र,"102,116",0
2,Only in constituency results,राष्ट्रिय जनमत पार्टी,953,0
3,Only in constituency results,नेपाल मानवतावादी पार्टी,949,0
4,Only in constituency results,राष्ट्रिय प्रजातन्त्र पार्टी नेपाल,315,0
5,Only in constituency results,नागरिक उन्मुक्ति पार्टी,271,0
6,Only in constituency results,राष्ट्रिय साझा पार्टी,191,0
7,Only in constituency results,नेपाल जनसेवा पार्टी,159,0
8,Only in constituency results,राष्ट्रिय सद्भावना पार्टी,109,0
9,Only in constituency results,युनाईटेड नेपाल डिमोक्रयाटीक पार्टी,32,0


## Key interpretation points

- The two vote systems are strongly aligned at the party level, but not identical.
- Large national parties dominate both, while the ratio and share-shift views expose where constituency performance runs ahead of or behind proportional support.
- Rank changes are small for the very biggest parties and more visible in the middle tier.
- A small number of unmatched parties remain, mostly because they appear only in one file or use labels that do not map cleanly across both sources.
